# Fleet Observability & Incident Root Cause Analysis (RCA)

This notebook contains the data analysis and root cause investigation for the Ati Motors AMR fleet operational incident on June 30, 2026, between 10:00 AM and 11:00 AM.

## 1. Setup and Libraries
We load standard packages and configure Matplotlib and Seaborn for plotting.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["font.size"] = 10

colors = {
    "Robot_01": "#1f77b4",
    "Robot_02": "#ff7f0e",
    "Robot_03": "#2ca02c",
    "Robot_04": "#d62728"
}

## 2. Load Data
We load the telemetry and trip databases.

In [ ]:
from IPython.display import display

telemetry = pd.read_csv("fleet_observability_assignment/telemetry.csv")
trips = pd.read_csv("fleet_observability_assignment/trips.csv")

print("Telemetry Records (Head):")
display(telemetry.head())

print("\nTrip Records:")
display(trips)

## 3. Log Exploration & Programmatic Duplicate Check
We read the operational log file `events.log` and programmatically identify duplicate log entries.

In [ ]:
with open("fleet_observability_assignment/events.log", "r") as f:
    lines = [line.strip() for line in f if line.strip()]

line_counts = Counter(lines)
duplicates = {line: count for line, count in line_counts.items() if count > 1}

print("Duplicate log entries found:")
for line, count in duplicates.items():
    print(f"  '{line}' appears {count} times")

## 4. Telemetry Data Quality Check
We examine the `trip_id` column inside `telemetry.csv` to verify its reliability.

In [ ]:
print("Unique trip IDs per timestamp in telemetry (first 5 groups):")
display(telemetry.groupby("timestamp")["trip_id"].unique().head(5))

print("\nRobots sharing a telemetry trip_id simultaneously:")
print(telemetry.groupby(["timestamp", "trip_id"])["robot_id"].count().unique())

## 5. Additional Observation (Not Directly Asked)

**Finding:** The `trip_id` field in telemetry.csv increments globally every 2 minutes for ALL robots simultaneously, unrelated to actual trip assignments in trips.csv.

**Why it matters:** If any downstream system (dashboard, reporting tool, alerting pipeline) joins telemetry data using this trip_id field instead of time-window matching against trips.csv, it will silently associate telemetry readings with the WRONG trip. This is a data-integrity bug that could cause future incidents to be misdiagnosed — e.g., blaming the wrong trip/robot for an anomaly. This should be flagged to the engineering team as a logging schema issue.

## 6. Chronological Data Merging
We perform a conditional time-window merge to map each telemetry poll to its corresponding active trip window from `trips.csv`.

In [ ]:
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"])

trips_clean = trips.copy()
trips_clean["start_dt"] = pd.to_datetime("2026-06-30 " + trips_clean["trip_start"])
trips_clean["end_dt"] = pd.to_datetime("2026-06-30 " + trips_clean["trip_end"].fillna("11:00:00"))

def match_trip(row):
    match = trips_clean[
        (trips_clean["robot_id"] == row["robot_id"]) &
        (row["timestamp"] >= trips_clean["start_dt"]) &
        (row["timestamp"] <= trips_clean["end_dt"])
    ]
    if not match.empty:
        return match.iloc[0]["trip_id"]
    return None

telemetry["actual_trip_id"] = telemetry.apply(match_trip, axis=1)

print("Sample of telemetry data with corrected actual trip associations:")
display(telemetry[telemetry["actual_trip_id"].notna()].head())

Note: Merged data (actual_trip_id) confirms only 17/120 telemetry rows fall within an official trip window. Robot_01's 1450ms latency spike (captured at 10:10:00) falls just outside T101's window (ended 10:06:20), since telemetry only polls every 2 minutes while the trip failure was instantaneous. This confirms the anomaly is real-time (from events.log) even though the periodic telemetry snapshot captured it slightly later.

## 7. Visualizations & Anomaly Detection
We plot network latency, CPU usage, battery charge, and spatial coordinates to identify fleet anomalies.

In [ ]:
plt.figure(figsize=(10, 4))
for r_id, group in telemetry.groupby("robot_id"):
    plt.plot(group["timestamp"], group["network_latency_ms"], label=r_id, marker='o', color=colors[r_id], linewidth=1.2)
plt.axhline(y=1000, color='r', linestyle='--', alpha=0.5, label="Latency Threshold (1000ms)")
plt.title("Network Latency (ms) over Time")
plt.xlabel("Time")
plt.ylabel("Latency (ms)")
plt.legend()
plt.tight_layout()
plt.show()

Robot_01 telemetry captured 1450ms at 10:10:00. This aligns with the real-time event log entry at 10:06:10 — the telemetry's 2-minute polling interval means the spike (which began before 10:06:10) was captured in the next scheduled snapshot.

In [ ]:
plt.figure(figsize=(10, 4))
for r_id, group in telemetry.groupby("robot_id"):
    plt.plot(group["timestamp"], group["cpu_usage"], label=r_id, marker='s', color=colors[r_id], linewidth=1.2)
plt.axhline(y=90, color='r', linestyle='--', alpha=0.5, label="CPU Threshold (90%)")
plt.title("CPU Usage (%) over Time")
plt.xlabel("Time")
plt.ylabel("CPU Usage (%)")
plt.legend()
plt.tight_layout()
plt.show()

Robot_02 telemetry captured 95% CPU usage at 10:24:00. This aligns with the real-time event log entry at 10:21:30 — the telemetry's 2-minute polling interval means the CPU spike (which began before 10:21:30) was captured in the next scheduled snapshot.

In [ ]:
plt.figure(figsize=(10, 4))
for r_id, group in telemetry.groupby("robot_id"):
    plt.plot(group["timestamp"], group["battery_level"], label=r_id, marker='^', color=colors[r_id], linewidth=1.2)
plt.title("Battery Level (%) over Time")
plt.xlabel("Time")
plt.ylabel("Battery (%)")
plt.legend()
plt.tight_layout()
plt.show()

Note: here the event log entry (10:44:00) is 28 minutes AFTER the telemetry captured the jump (10:16:00) — opposite pattern from above. This could indicate the log was queued/delayed on this specific robot.

In [ ]:
plt.figure(figsize=(6, 6))
for r_id, group in telemetry.groupby("robot_id"):
    valid = group[(group["position_x"] < 100) & (group["position_y"] < 100)]
    plt.plot(valid["position_x"], valid["position_y"], label=r_id, marker='o', markersize=3, color=colors[r_id], alpha=0.5)

plt.scatter([999], [999], color='red', s=100, label="Robot_03 Out-of-Bounds")
plt.title("Fleet 2D Spatial Trajectories")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.legend()
plt.tight_layout()
plt.show()

Robot_03 position coordinates recorded an out-of-bounds coordinate jump of (999, 999) at 10:18:00, which aligns with the log event at 10:18:05.

## 8. Table of Identified Fleet Anomalies

| Anomaly Type | Robot ID | Category | Finding | Reason | Impact |
|---|---|---|---|---|---|
| Latency Spike | Robot_01 | **Network** | 1450ms at 10:10:00 | Avg ~130ms | Trip cancellation |
| False Battery Warning | Robot_01 | **System/Software** | Warning at 76% battery | False trigger | No real impact |
| CPU Spike | Robot_02 | **Robot Behavior** | 95% at 10:24:00 | Exceeds 30-70% norm | Temporary overload |
| Position Jump | Robot_03 | **Robot Behavior** (sensor) | (999,999) | Outside 0-50 range | GPS glitch |
| Battery Jump | Robot_04 | **Robot Behavior** (sensor) | +9% no charge | Physically impossible | Calibration issue |
| Stuck Trip | Robot_04 | **System/Software** | No completion event | Trip left open | DB integrity risk |
| Duplicate Log | Robot_02 | **System/Software** | TripStarted logged twice | Redundant write | Logging bug |

## 9. Incident Chronological Timeline (10:00 AM - 11:00 AM)

- **10:00:00**: Fleet monitoring begins.
- **10:05:12**: **Robot_01** starts trip `T101` from A to B.
- **10:05:14**: **Robot_01** successfully acquires spatial lock for `Zone_A`.
- **10:05:55**: **Robot_01** triggers a `BatteryLowWarning` (false alarm, battery is at 76%).
- **10:06:10**: **Robot_01** experiences a massive **Network Latency Spike (1450ms)**.
- **10:06:18**: Due to high latency, **Robot_01** fails to communicate with the lock manager to acquire the lock for `Zone_B` within the safety timeout period, triggering `ResourceLockTimeout Zone_B`.
- **10:06:20**: Lacking the spatial lock to enter Zone_B and experiencing bad connection, **Robot_01** cancels its trip `T101` (status: `cancelled`).
- **10:15:00**: **Robot_02** starts trip `T102` (duplicate log entries recorded in events.log).
- **10:16:00**: **Robot_04** battery charge jumps from 32% to 41% (shows in events.log at 10:44:00 due to local queue lag).
- **10:17:40**: **Robot_03** starts trip `T204`.
- **10:17:42**: **Robot_03** acquires lock for Zone_C.
- **10:18:00**: **Robot_03** telemetry records out-of-bounds coordinates `(999, 999)`.
- **10:18:05**: **Robot_03** log records `PositionJumpDetected`.
- **10:18:15**: **Robot_03** successfully completes trip `T204`.
- **10:21:30**: **Robot_02** experiences CPU High 95% (shows in telemetry at 10:24:00 due to periodic 2-minute polling delay).
- **10:25:00**: **Robot_02** successfully completes trip `T102`.
- **10:40:00**: **Robot_04** starts trip `T301` from M to N.
- **10:58:00**: Fleet monitoring ends. Robot_04's trip remains in a running state.

## 10. Root Cause Analysis (RCA) & Conclusions

### Most Critically Affected Robot
**Robot_01** was the most critically affected robot. It is the only vehicle whose trip aborted and was cancelled due to system failure. The other robots completed their trips successfully (Robot_02 and Robot_03) or continued operating and reporting correct telemetry coordinates (Robot_04).

### Root Cause Determination
The primary root cause of the incident is **Network Conditions** (specifically the **1450ms network latency spike**).

**Evaluation of Categories**:
1. **Network Conditions (Accepted)**: Robot_01's 1450ms latency spike directly caused the trip cancellation. High latency blocked real-time spatial coordinate heartbeat signals. Without active communication, the robot could not request or renew the lock for Zone_B before the timeout expired, triggering `ResourceLockTimeout Zone_B` and forcing a safety abort.
2. **Robot Behavior (Rejected)**: Robot behavior anomalies (Robot_03's out-of-bounds coordinate jump and Robot_04's +9% battery level jump) were sensor-reporting glitches that did not cause trip cancellations or prevent successful mission completion.
3. **System Behavior (Rejected)**: System and software anomalies (Robot_02's 95% CPU spike and duplicate log entries) did not cause any trip failures. Robot_02 successfully completed its trip `T102`.

5 out of 7 anomalies are Robot Behavior or System-related, but only the Network latency anomaly directly correlates with the actual failure (trip cancellation), which is why Network Conditions is identified as the primary root cause.

### Log Queuing Delay
We observe one instance (Robot_04) where the event log entry lags telemetry by 28 minutes — this is a single data point and should be treated as an isolated observation, not a systemic pattern, since the Robot_01 and Robot_02 events show the opposite timing relationship.

## 11. Assumptions

- Incident date assumed as 2026-06-30 for constructing full timestamps from trip_start/trip_end.
- Robot_04's trip (T301) has no recorded trip_end; we assumed the monitoring window end (11:00:00) as a placeholder for merge purposes only — this does NOT mean the trip actually ended then.
- Where an event log timestamp doesn't exactly match a telemetry sample (e.g. BatteryLowWarning at 10:05:55), the nearest telemetry reading (10:06:00) was used as an approximation.
- The telemetry.csv `trip_id` field was treated as unreliable/buggy and excluded from analysis (see Section 4).